# Forget-MI LoKU — Machine Unlearning Pipeline

> ✅ **Cell 4 đang SWEEP IHL {0.5, 0.75, 1.0} (exp11c, no F_re) — không cần sửa tay.**
> Đã `git push` code+config mới? → Colab: **Cell 1** (pull) →
> *(bỏ qua Cell 2, 3 nếu đã có data)* → **Cell 3.5 → Cell 4 → Cell 5**.
> Cell 4 chạy 3 giá trị IHL, in bảng `forget_ce` vs `test_ce` để chọn sweet-spot
> (IHL lớn nhất mà forget_ce ≲ test_ce). Đổi `IHLS`/`EXP_NAME` ở đầu Cell 4 nếu cần.

Notebook thực hiện toàn bộ pipeline:

1. **Cell 1** — Mount Drive + pull code mới (KHÔNG xóa data đã extract)
2. **Cell 2** — Extract data & models (chỉ chạy LẦN ĐẦU)
3. **Cell 3** — Preprocess (chỉ chạy LẦN ĐẦU)
4. **Cell 3.5** — Verify config (mỗi lần chạy exp mới)
5. **Cell 4** — Huấn luyện LoKU Unlearning (đang ở chế độ sweep IHL)
6. **Cell 5** — **Auto-commit & push** kết quả lên GitHub

## Workflow cho mỗi experiment mới

### Lần đầu chạy (full setup)

1. Sửa `config.yaml` ở local → push lên GitHub
2. Mở Colab → chạy **Cell 1 → Cell 2 → Cell 3** (lần đầu, đợi ~3 phút)
3. (Tùy chọn) chỉnh `EXP_NAME` / `IHLS` ở đầu Cell 4
4. Chạy **Cell 3.5 → Cell 4 → Cell 5**

### Lần thứ 2 trở đi (skip extract)

1. Sửa `config.yaml` ở local → push lên GitHub
2. Colab: **Cell 1** (pull code mới, ~10s)
3. **BỎ QUA Cell 2 + Cell 3** (data đã có)
4. (Tùy chọn) chỉnh `EXP_NAME` / `IHLS` ở Cell 4
5. **Cell 3.5 → Cell 4 → Cell 5**

### Sau khi Colab xong

6. Local: `git pull` để lấy file MD + summary về
7. Điền 3 section vào file MD per-run nếu cần: Observations / Conclusion / Next steps
8. `git push`

> Lần đầu setup Cell 5: tạo file `/content/drive/MyDrive/Forget-MI-Project/.git-secrets.json` (xem hướng dẫn trong cell).

In [13]:
# ====================================
# CELL 1: Kết nối Drive & Pull Code (giữ data, không clone lại)
# ====================================
from google.colab import drive
import os

# 1. Mount Google Drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive', force_remount=True)
else:
    print("✅ Google Drive đã được kết nối!")

# 2. Pull code mới (KHÔNG xóa thư mục → data đã extract được giữ nguyên)
%cd /content
REPO = "Forget-MI-LoKU"
REPO_URL = "https://github.com/nhnhu146/Forget-MI-LoKU.git"

if not os.path.exists(REPO):
    print(f"🔽 Clone lần đầu: {REPO}")
    !git clone {REPO_URL}
else:
    print(f"🔄 Pull code mới (giữ data đã extract)")
    %cd {REPO}
    !git fetch origin
    !git reset --hard origin/master 2>&1 | tail -3
    %cd /content

%cd {REPO}
!git log --oneline -1

# 3. Cài đặt thư viện (chỉ chạy lần đầu hoặc khi cần update)
import importlib.util
need_install = importlib.util.find_spec("peft") is None or importlib.util.find_spec("pydicom") is None
if need_install:
    print("📦 Cài đặt thư viện...")
    !pip install -q pydicom scikit-image wandb pyyaml pandas
    !pip install -q "transformers==4.38.0" "peft==0.10.0" "accelerate==0.27.0"
else:
    print("✅ Thư viện đã cài, bỏ qua.")

print("\n✅ Môi trường và mã nguồn đã sẵn sàng!")
print("ℹ️  Lần đầu: chạy Cell 2 → Cell 3 (extract data).")
print("ℹ️  Lần sau: bỏ qua Cell 2 + Cell 3, đi thẳng Cell 3.5 → Cell 4 → Cell 5.")

In [14]:
# ====================================
# CELL 2: Giải nén Data & Models
# ====================================
!python setup_data.py

In [15]:
# ====================================
# CELL 3: Tiền xử lý & Thiết lập Output
# ====================================
# CHỈ CHẠY LẦN ĐẦU (sau đó cache features được giữ trong /content/.../data/metadata/)
import os
import shutil

# 1. Tạo all_data.tsv từ các file báo cáo (idempotent)
!python make_tsv.py

# 2. Cache features — KHÔNG xóa nữa (xóa = phải regenerate 5+ phút mỗi lần)
#    Uncomment 2 dòng dưới nếu bạn THỰC SỰ muốn force regenerate cache
# !rm -f ./data/metadata/cachedfeatures_train_seqlen-*
# !rm -f ./data/metadata/cachednoisyfeatures_train_seqlen-*

# 3. Kết nối thư mục Output với Drive để lưu bền vững
DRIVE_RESULTS = "/content/drive/MyDrive/Forget-MI-Project/unlearning_output"
os.makedirs(DRIVE_RESULTS, exist_ok=True)

if os.path.exists("unlearning_output"):
    if os.path.islink("unlearning_output"):
        os.unlink("unlearning_output")
    else:
        shutil.rmtree("unlearning_output")

!ln -s "{DRIVE_RESULTS}" ./unlearning_output

# 4. Verify cache files có sẵn không
cache_dir = "./data/metadata"
has_cache = False
if os.path.exists(cache_dir):
    files = os.listdir(cache_dir)
    has_cache = any(f.startswith(("cachedfeatures_train_seqlen", "cachednoisyfeatures_train_seqlen"))
                    for f in files)

print(f"\n✅ Output sẽ được lưu tại: {DRIVE_RESULTS}")
print(f"{'✅' if has_cache else '⚠️ '} Cache features {'đã có' if has_cache else 'CHƯA có'} trong {cache_dir}")
if not has_cache:
    print("   → Cell 4 lần đầu sẽ chậm (~5 phút regenerate features). Lần sau sẽ load cache nhanh.")

In [ ]:
# ====================================
# CELL 3.5: Verify config — confirm code mới nhất từ GitHub
# ====================================
# Chạy cell này TRƯỚC Cell 4 để chắc chắn config đúng với exp đang định chạy.
# Nếu thấy giá trị CŨ → bạn quên push từ local, hãy push rồi rerun Cell 1.

print("📋 Config hiện tại (các tham số hay đổi giữa các exp):\n")
!grep -E "^\s*(forget_margin|eta_re_anchor|alpha|beta|theta|gamma|lora_r|lora_alpha|lora_target_modules|use_noise|unlearn_epochs|learning_rate|kappa_cls_retain|kappa_cls_forget|cls_forget_clamp|unfreeze_classifier_heads|uniform_prior_weight|distill_teacher|distill_retain_weight|distill_forget_weight|distill_temperature|loku_subtract_scale|ihl_forget_weight|lora_image_last_k_blocks|lora_image_include_fc1|loku_image_subtract_scale):" -A 1 config.yaml | grep -v "^--"

print("\n🧪 Kiểm tra code có fix Exp 03+ / image-FILA / exp11 (teacher=og):")
!grep -c "Unfroze classifier\|L_cls_ret\|L_cls_frg\|L_distill_ret\|L_distill_frg\|L_ihl\|TRUE-SUBTRACTION\|resolve_image_targets\|_fila_decompose\|use_og_teacher\|distill_teacher" training/forgetmi_loku.py | xargs -I{} echo "   → Số lần khớp pattern fix: {} (nên >= 13)"

print("\n🖼️  exp10+ sanity — image PEFT BẬT & teacher distill:")
!grep -E "^\s*(lora_image_last_k_blocks|distill_teacher|ihl_forget_weight):" -A 1 config.yaml | grep "value:" | xargs -I{} echo "   → {}"

print("\n🔍 Git commit đang chạy:")
!git log --oneline -1

print("\n👉 Nếu config KHÔNG đúng với exp bạn định chạy:")
print("   1. Local: kiểm tra `git status` xem đã commit chưa")
print("   2. Local: `git push`")
print("   3. Colab: chạy lại Cell 1 (clone fresh)")
print("   4. Chạy lại Cell 3.5 này để verify")

In [ ]:
# ====================================
# CELL 4: SWEEP IHL {0.5, 0.75, 1.0} — tìm sweet-spot (single seed 42)
# ====================================
# ✅ exp11c đã điền sẵn — chỉ bấm chạy 1 lần. Mỗi IHL = 1 lần train+eval (~0.2h) → ~0.6h.
#    Mục tiêu: chọn IHL LỚN nhất mà forget_ce ≈ test_ce (forget KHÔNG vượt test = không over-forget).
import os, numpy as np, pandas as pd

EXP_NAME = "exp11c_ihl_sweep"
IHLS     = [0.5, 0.75, 1.0]                 # đổi/thêm giá trị tùy ý
HYPOTHESIS = ("exp11c sweep IHL de tim sweet-spot (no F_re). IHL=0.5 forget_ce(1.57)<test(1.74) -> con du dia, "
              "Forget-AUC 0.758 hoi cao. Tang IHL de Forget-AUC giam ma giu forget_ce <= test_ce (MIA lanh manh). "
              "Chon IHL lon nhat con thoa forget_ce ~ test_ce.")

CSV = "unlearning_output/results_summary.csv"
if os.path.exists(CSV):
    os.remove(CSV)        # fresh → tổng hợp chỉ thấy các IHL của lần này

# ----- Sweep từng IHL (mỗi lần override config, --fresh để độc lập) -----
for i, v in enumerate(IHLS):
    tag = f"ihl{int(round(v*100)):03d}"     # 0.5→ihl050, 0.75→ihl075, 1.0→ihl100
    print(f"\n{'='*60}\n🔧 IHL = {v}   ({i+1}/{len(IHLS)})\n{'='*60}")
    cmd = (f'PYTHONPATH=. WANDB_MODE=disabled python training/forgetmi_loku.py '
           f'--config config.yaml --fresh --override "ihl_forget_weight={v}" '
           f'--exp {EXP_NAME}_{tag} --hypothesis "{HYPOTHESIS}"')
    get_ipython().system(cmd)

# ----- Bảng tổng hợp theo IHL -----
print(f"\n{'='*72}\n📊 SWEEP IHL  (forget_ce nên ≈ test_ce; KHÔNG vượt = không over-forget)\n{'='*72}")
df = pd.read_csv(CSV)
cols = [('ihl','IHL'), ('forget_ce','forget_ce'), ('test_ce','test_ce'),
        ('MIA','MIA_ps'), ('MIA_paper','MIA_pp'),
        ('Df_AUC','Df_AUC'), ('Df_F1','Df_F1'), ('Dt_AUC','Dt_AUC'), ('Dt_F1','Dt_F1')]
cols = [(k, h) for k, h in cols if k in df.columns]
hdr = "".join(f"{h:>10}" for _, h in cols)
print(hdr); print("-" * len(hdr))
md_rows = ["| " + " | ".join(h for _, h in cols) + " | over-forget? |",
           "|" + "---|" * (len(cols) + 1)]
for v in IHLS:
    sub = df[np.isclose(df['ihl'], v)]
    if sub.empty:
        continue
    r = sub.iloc[-1]
    over = "⚠️ YES" if (r.get('test_ce', 0) > 0 and r.get('forget_ce', 0) > 1.3 * r.get('test_ce', 1)) else "no"
    print("".join(f"{(r[k] if k!='ihl' else v):>10.3f}" for k, _ in cols))
    md_rows.append("| " + " | ".join(f"{(r[k] if k!='ihl' else v):.3f}" for k, _ in cols) + f" | {over} |")

out_md = f"experiments/{EXP_NAME}_summary.md"
with open(out_md, "w", encoding="utf-8") as f:
    f.write(f"# IHL sweep — {EXP_NAME}\n\nIHL values: {IHLS} (single seed 42, no F_re)\n\n"
            + "\n".join(md_rows)
            + "\n\n**Paper (3%)**: MIA=0.571 | Df_AUC=0.735 | Df_F1=0.393 | Dt_AUC=0.625 | Dt_F1=0.250\n"
            + "**Sweet-spot**: chọn IHL lớn nhất mà forget_ce ≲ test_ce (Forget-AUC thấp nhất, MIA lành mạnh).\n")

print("\n" + "=" * 72)
print(f"💾 Summary: {out_md}")
print("👉 Chọn IHL sweet-spot → đặt vào config.yaml → chạy multi-seed. Rồi CELL 5 để push.")

In [18]:
# ====================================
# CELL 5: Auto-commit & push experiment results lên GitHub
# ====================================
# 3 cách setup credentials (chọn 1, theo độ tiện):
#
# CÁCH A — Lưu vào Drive (KHUYẾN NGHỊ, setup 1 lần dùng mãi):
#   Tạo file /content/drive/MyDrive/Forget-MI-Project/.git-secrets.json
#   với nội dung:
#   {
#     "GITHUB_TOKEN": "ghp_xxxxxxxxxxxx",
#     "GIT_EMAIL": "ban@gmail.com",
#     "GIT_NAME": "Nguyen Hoang Nhu"
#   }
#   Tạo token tại: https://github.com/settings/tokens (scope: repo)
#
# CÁCH B — Colab Secrets (CHỈ web colab.research.google.com):
#   Click 🔑 ở sidebar → Add secret: GITHUB_TOKEN, GIT_EMAIL, GIT_NAME
#
# CÁCH C — Nhập tay mỗi session (lazy, không cần setup):
#   Bỏ qua A và B → cell sẽ tự hỏi token mỗi lần chạy
# ===========================================================
import os, json, getpass
from pathlib import Path

GITHUB_REPO = "nhnhu146/Forget-MI-LoKU"
BRANCH = "master"

def load_secrets():
    # CÁCH A — file trên Drive
    drive_path = Path("/content/drive/MyDrive/Forget-MI-Project/.git-secrets.json")
    if drive_path.exists():
        s = json.loads(drive_path.read_text())
        print(f"🔑 Đã load credentials từ {drive_path}")
        return s.get('GITHUB_TOKEN'), s.get('GIT_EMAIL'), s.get('GIT_NAME')

    # CÁCH B — Colab Secrets (web Colab)
    try:
        from google.colab import userdata
        t = userdata.get('GITHUB_TOKEN')
        if t:
            print("🔑 Đã load credentials từ Colab Secrets")
            return t, userdata.get('GIT_EMAIL'), userdata.get('GIT_NAME')
    except Exception:
        pass

    # CÁCH B2 — environment variables
    if os.environ.get('GITHUB_TOKEN'):
        print("🔑 Đã load credentials từ env vars")
        return (os.environ['GITHUB_TOKEN'],
                os.environ.get('GIT_EMAIL', ''),
                os.environ.get('GIT_NAME', ''))

    # CÁCH C — nhập tay (fallback)
    print("🔑 Nhập credentials thủ công (sẽ ẩn khi gõ token):")
    print("   (lần sau muốn auto, tạo file Drive theo CÁCH A ở comment trên)")
    t = getpass.getpass("  GitHub token (ghp_...): ").strip()
    e = input("  Git email: ").strip()
    n = input("  Git name:  ").strip()
    return t, e, n


TOKEN, EMAIL, NAME = load_secrets()

if TOKEN and EMAIL and NAME:
    # 1. Configure git identity (chỉ trong repo này, không ảnh hưởng global)
    !git config user.email "{EMAIL}"
    !git config user.name "{NAME}"

    # 2. Inject token vào remote URL (chỉ trong session này)
    !git remote set-url origin https://{TOKEN}@github.com/{GITHUB_REPO}.git

    # 3. Pull trước để tránh conflict
    !git pull --rebase origin {BRANCH} 2>&1 | tail -5

    # 4. Stage CHỈ file experiment
    !git add experiments/ 2>/dev/null

    # 5. Hiển thị thay đổi
    changes = !git diff --cached --name-only
    if changes and any(c.strip() for c in changes):
        print("\n📦 Files sẽ commit:")
        for f in changes:
            if f.strip():
                print(f"   - {f}")

        commit_msg = f"exp {EXP_NAME}: auto-tracked results"
        !git commit -m "{commit_msg}"
        !git push origin {BRANCH}

        print(f"\n✅ Đã push lên GitHub")
        print(f"🔗 Xem online: https://github.com/{GITHUB_REPO}/tree/{BRANCH}/experiments")
    else:
        print("ℹ️  Không có file experiment mới để commit.")
else:
    print("⚠️  Thiếu credentials — bỏ qua push. Setup theo CÁCH A/B/C ở comment trên.")